In [1]:
import os
import argparse
import random
import time
import warnings
warnings.filterwarnings("ignore")
 
import h5py
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision.transforms.functional import rotate as tvrotate
from torchvision import transforms
from tqdm.auto import tqdm

def set_seed(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
 
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
def get_args():
    p = argparse.ArgumentParser(description="Bonus Task – Exp 3: Lie Canonicalization")
    p.add_argument("--task1_ckpt",   type=str, default="/kaggle/input/models/vitthal2945/e2e-vaemodel/pytorch/default/1/best_model.pt")
    p.add_argument("--task2_ckpt",   type=str, default="/kaggle/input/models/vitthal2945/best-model-task2/pytorch/default/1/best_model_task2.pt")
    p.add_argument("--clf_ckpt",     type=str, default="/kaggle/input/models/vitthal2945/classifier-exp1/pytorch/default/1/classifiers_exp1.pt")
    p.add_argument("--data_dir",     type=str, default="/kaggle/input/datasets/vitthal2945/e2e-hidden-symmetries-dataset")
    p.add_argument("--out_dir",      type=str, default="./bonus_outputs/exp3")
    p.add_argument("--latent_dim",   type=int, default=16)
    p.add_argument("--n_classes",    type=int, default=2)
    p.add_argument("--n_canon_steps", type=int, default=5,
                   help="Gradient descent steps for energy minimisation (CLC)")
    p.add_argument("--canon_lr",     type=float, default=0.05,
                   help="Learning rate for canonical angle optimisation")
    p.add_argument("--n_prototype_samples", type=int, default=200,
                   help="Number of 0° samples used to estimate canonical direction v_c")
    return p.parse_args([])

In [3]:
class ResBlock(nn.Module):
    def __init__(self, ch, g=8):
        super().__init__()
        g = min(g, ch)
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.GroupNorm(g, ch), nn.SiLU(),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.GroupNorm(g, ch),
        )
        self.act = nn.SiLU()
    def forward(self, x): return self.act(x + self.net(x))
 
class Encoder(nn.Module):
    def __init__(self, ld):
        super().__init__()
        self.stem  = nn.Sequential(nn.Conv2d(1, 32, 4, 2, 1, bias=False), nn.GroupNorm(8, 32), nn.SiLU())
        self.l1    = nn.Sequential(ResBlock(32),  nn.Conv2d(32,  64, 4, 2, 1, bias=False), nn.GroupNorm(8,  64), nn.SiLU())
        self.l2    = nn.Sequential(ResBlock(64),  nn.Conv2d(64, 128, 3, 2, 1, bias=False), nn.GroupNorm(8, 128), nn.SiLU())
        self.l3    = ResBlock(128)
        self.fc    = nn.Sequential(nn.Flatten(), nn.Linear(128*4*4, 512), nn.SiLU())
        self.fc_mu = nn.Linear(512, ld); self.fc_lv = nn.Linear(512, ld)
    def forward(self, x):
        h = self.l3(self.l2(self.l1(self.stem(x)))); h = self.fc(h)
        return self.fc_mu(h), self.fc_lv(h)
 
class Decoder(nn.Module):
    def __init__(self, ld):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(ld, 512), nn.SiLU(), nn.Linear(512, 128*4*4), nn.SiLU())
        self.u1 = nn.Sequential(ResBlock(128), nn.ConvTranspose2d(128, 64, 3, 2, 1, output_padding=1), nn.GroupNorm(8,  64), nn.SiLU())
        self.u2 = nn.Sequential(ResBlock(64),  nn.ConvTranspose2d(64,  32, 4, 2, 1),                  nn.GroupNorm(8,  32), nn.SiLU())
        self.u3 = nn.Sequential(ResBlock(32),  nn.ConvTranspose2d(32,   1, 4, 2, 3),                  nn.Sigmoid())
    def forward(self, z): return self.u3(self.u2(self.u1(self.fc(z).view(-1, 128, 4, 4))))
 
class Task1VAE(nn.Module):
    def __init__(self, ld):
        super().__init__()
        self.encoder = Encoder(ld); self.decoder = Decoder(ld)
    def encode(self, x): mu, _ = self.encoder(x); return mu
    def decode(self, z): return self.decoder(z)

In [4]:
class LieAlgebraModule(nn.Module):
    """
    Exact replica of Task-2 LieAlgebraModule.
    A = L - L^T (skew-symmetric), R(θ) = expm(θ·A).
    """
    def __init__(self, ld: int):
        super().__init__()
        self.ld      = ld
        self.L_lower = nn.Parameter(torch.zeros(ld, ld))
 
    def get_A(self) -> torch.Tensor:
        L = torch.tril(self.L_lower, diagonal=-1)
        return L - L.T
 
    def R(self, theta_rad: float) -> torch.Tensor:
        return torch.linalg.matrix_exp(theta_rad * self.get_A())
 
    def forward(self, z: torch.Tensor, theta_rad: float) -> torch.Tensor:
        return z @ self.R(theta_rad).T
 
    def inverse(self, z: torch.Tensor, theta_rad: float) -> torch.Tensor:
        return z @ self.R(-theta_rad).T
 
 
class ResidualCorrection(nn.Module):
    ANGLE_FEATS = 4
    def __init__(self, ld: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(ld + self.ANGLE_FEATS, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, ld),
        )
 
    @staticmethod
    def angle_features(theta_rad: float, B: int, dev) -> torch.Tensor:
        th = torch.tensor(theta_rad, dtype=torch.float32)
        feats = torch.stack([th.sin(), th.cos(), (2*th).sin(), (2*th).cos()])
        return feats.unsqueeze(0).expand(B, -1).to(dev)
 
    def forward(self, z: torch.Tensor, theta_rad: float) -> torch.Tensor:
        ang = self.angle_features(theta_rad, z.size(0), z.device)
        return self.net(torch.cat([z, ang], dim=1))
 
 
class HybridLieTransform(nn.Module):
    """Exact replica of Task-2 HybridLieTransform."""
    def __init__(self, ld: int):
        super().__init__()
        self.lie = LieAlgebraModule(ld)
        self.res = ResidualCorrection(ld)
 
    def forward(self, z: torch.Tensor, theta_rad: float) -> torch.Tensor:
        return self.lie(z, theta_rad) + self.res(z, theta_rad)
 
    def inverse_approx(self, z: torch.Tensor, theta_rad: float, n_iters: int = 4) -> torch.Tensor:
        z_k = self.lie.inverse(z, theta_rad)
        for _ in range(n_iters):
            delta = self.res(z_k, theta_rad)
            z_k   = self.lie.inverse(z - delta, theta_rad)
        return z_k
 
 
class MLPClassifier(nn.Module):
    def __init__(self, ld: int, nc: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(ld, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, nc),
        )
    def forward(self, z): return self.net(z)

In [5]:
def load_vae(path, ld):
    vae = Task1VAE(ld).to(device)
    if os.path.exists(path):
        ck = torch.load(path, map_location=device, weights_only=False)
        vae.load_state_dict(ck.get("model", ck))
        print(f"  ✓ VAE loaded (epoch {ck.get('epoch', '?')})")
    else:
        print(f"  ⚠  VAE ckpt not found: {path}")
    vae.eval()
    for p in vae.parameters(): p.requires_grad = False
    return vae
 
def load_hybrid_transform(path, ld):
    T_net = HybridLieTransform(ld).to(device)

    if os.path.exists(path):
        ck = torch.load(path, map_location=device)

        # Handle different checkpoint formats
        if "T_net" in ck:
            sd = ck["T_net"]
        elif "model" in ck:
            sd = ck["model"]
        else:
            sd = ck

        T_net.load_state_dict(sd, strict=False)

        print("✓ HybridLieTransform loaded correctly")

    else:
        print(f"⚠ Task-2 ckpt not found: {path}")

    T_net.eval()
    for p in T_net.parameters():
        p.requires_grad = False

    return T_net
 
def load_clf(path, ld, nc, key="noaug"):
    clf = MLPClassifier(ld, nc).to(device)
    if os.path.exists(path):
        ck  = torch.load(path, map_location=device, weights_only=False)
        clf.load_state_dict(ck.get(key, ck))
        print(f"  ✓ Classifier ('{key}') loaded")
    else:
        print(f"  ⚠  Classifier ckpt not found: {path}")
    clf.eval()
    for p in clf.parameters(): p.requires_grad = False
    return clf
 
def load_h5(path):
    print(f"  loading {os.path.basename(path)} ...", end=" ", flush=True)
    t0 = time.time()
    with h5py.File(path, "r") as f:
        imgs   = torch.from_numpy(f["images"][:].astype("float32")).unsqueeze(1).clamp(0., 1.)
        labels = torch.from_numpy(f["labels"][:].astype("int64"))
        angles = torch.from_numpy(f["angles"][:].astype("int64"))
    print(f"done ({time.time()-t0:.1f}s)")
    return imgs, labels, angles

In [6]:
@torch.no_grad()
def compute_canonical_direction(
    vae:    Task1VAE,
    imgs_0: torch.Tensor,
    n:      int = 200
) -> torch.Tensor:
    """
    v_c = normalise(mean_z { z : image is at 0° })
    This defines the canonical orientation in latent space.
    """
    idx = torch.randperm(len(imgs_0))[:n]
    Z   = []
    for i in range(0, len(idx), 128):
        Z.append(vae.encode(imgs_0[idx[i:i+128]].to(device)).cpu())
    Z = torch.cat(Z)  # (n, ld)
    v_c = Z.mean(dim=0)
    return F.normalize(v_c, dim=0).to(device)  # (ld,)

In [7]:
def canonical_energy(
    z_canon: torch.Tensor,
    v_c:     torch.Tensor
) -> torch.Tensor:
    """
    E(t) = 1 - cos(z(t), v_c)  ∈ [0, 2]
    Minimum at t* where z(t*) aligns maximally with the canonical direction.
    """
    z_norm = F.normalize(z_canon, dim=-1)
    v_norm = F.normalize(v_c.unsqueeze(0).expand_as(z_norm), dim=-1)
    cosine = (z_norm * v_norm).sum(dim=-1)
    return (1.0 - cosine).mean()
 
 
def lie_canonicalize_batch(
    z0:       torch.Tensor,      # (B, ld) latent codes to canonicalise
    T_net:    HybridLieTransform,
    v_c:      torch.Tensor,      # (ld,) canonical direction
    n_steps:  int = 50,
    lr:       float = 0.05,
    t_init:   float = 0.0,
) -> tuple:
    """
    Continuous Lie Canonicalization (CLC).
 
    Solves: t* = argmin_t E(T(z0, t))
    using Adam gradient descent on the scalar rotation angle t.
 
    Returns:
        z_canon:  (B, ld)  canonicalized latent codes
        t_star:   (B,)     optimal rotation angles found
        energies: (n_steps,) energy values during optimization (for first sample)
    """
    B = z0.shape[0]
    # Learnable angle parameter per sample
    t_param = torch.full((B,), t_init, requires_grad=True, device=device)
    opt     = torch.optim.Adam([t_param], lr=lr)
    energies = []
 
    for step in range(n_steps):
        opt.zero_grad()
        # Apply T_net with per-sample angle — vectorize by iterating
        # For efficiency, group by unique rounded angles
        z_t_list = []
        for b in range(B):
            theta = t_param[b]  # scalar, has grad
        
            # Linear Lie part
            A       = T_net.lie.get_A()
            R_theta = torch.linalg.matrix_exp(theta * A)
            z_lie   = z0[b:b+1] @ R_theta.T
        
            # Residual correction (keep differentiable)
            z_res = T_net.res(z0[b:b+1], theta)
        
            z_t_list.append(z_lie + z_res)
        z_t   = torch.cat(z_t_list, dim=0)   # (B, ld)
        E     = canonical_energy(z_t, v_c)
        E.backward()
        opt.step()
        # Clamp t to [-π, π] to keep it within one full rotation
        with torch.no_grad():
            t_param.clamp_(-np.pi, np.pi)
        if B > 0:
            energies.append(E.item())
 
    # Final forward pass (no grad)
    with torch.no_grad():
        z_canon_list = []
        for b in range(B):
            theta_val = t_param[b].item()
            z_c = T_net(z0[b:b+1], theta_val)
            z_canon_list.append(z_c)
        z_canon = torch.cat(z_canon_list, dim=0)
        t_star  = t_param.detach()
 
    return z_canon, t_star, energies
 
 
@torch.no_grad()
def grid_canonicalize_batch(
    z0:    torch.Tensor,
    T_net: HybridLieTransform,
    v_c:   torch.Tensor,
    angles_deg: list = None
) -> torch.Tensor:
    """
    Grid search canonicalization: try all quantized rotation angles,
    pick the one that maximally aligns with v_c.
    This is the discrete ablation of CLC.
    """
    if angles_deg is None:
        angles_deg = [i * 30 for i in range(12)]
    best_z    = z0.clone()
    best_cos  = -torch.ones(len(z0), device=device)
 
    for deg in angles_deg:
        theta_rad = np.radians(float(deg))
        z_t   = T_net(z0, theta_rad)              # (B, ld)
        cos_t = F.normalize(z_t, dim=1) @ F.normalize(v_c, dim=0)  # (B,)
        improve = cos_t > best_cos
        best_z[improve]   = z_t[improve]
        best_cos[improve] = cos_t[improve]
    return best_z

In [8]:
def evaluate_canon_strategy(
    strategy_fn,      # (imgs_chunk) → z_canon (B, ld)
    clf:   MLPClassifier,
    vae:   Task1VAE,
    imgs_0: torch.Tensor,
    lbls_0: torch.Tensor,
    angles: list,
    dig2cls: dict,
    batch:  int = 64,
    desc:   str = ""
) -> dict:
    results = {}
    for angle in tqdm(angles, desc=f"  eval {desc}", leave=False):
        correct = total = 0
        for i in range(0, len(imgs_0), batch):
            chunk = imgs_0[i:i+batch]
            if angle != 0:
                chunk = tvrotate(chunk, float(angle),
                                 interpolation=transforms.InterpolationMode.BILINEAR, fill=[0.])
            z_canon = strategy_fn(chunk.to(device))     # (B, ld)
            with torch.no_grad():
                pred = clf(z_canon).argmax(dim=1).cpu()
            lbl_chunk = torch.tensor([dig2cls.get(l.item(), l.item())
                                      for l in lbls_0[i:i+batch]])
            correct += (pred == lbl_chunk).sum().item()
            total   += len(pred)
        results[angle] = correct / total
    return results
 
 

def canonicalization_quality_metric(
    T_net:   HybridLieTransform,
    vae:     Task1VAE,
    imgs_0:  torch.Tensor,
    v_c:     torch.Tensor,
    n_steps: int,
    lr:      float,
    batch:   int = 64
) -> tuple:
    """
    CQM = mean_z cos(z_canon(z_θ), v_c)  measured across random θ.
    Higher = better canonicalization.
    Also returns CQM_grid for comparison.
    """
    test_angles = [0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330]
    rng         = np.random.default_rng(99)
    sample_idx  = rng.choice(len(imgs_0), min(200, len(imgs_0)), replace=False)
    cqm_clc_vals   = []
    cqm_grid_vals  = []
    for angle in test_angles:
        chunk = imgs_0[sample_idx[:batch]]
        if angle != 0:
            chunk = tvrotate(chunk, float(angle),
                             interpolation=transforms.InterpolationMode.BILINEAR, fill=[0.])
        with torch.no_grad():
            z0 = vae.encode(chunk.to(device))
        # CLC
        z_clc, _, _ = lie_canonicalize_batch(z0, T_net, v_c, n_steps, lr)
        cos_clc      = (F.normalize(z_clc, dim=1) @ F.normalize(v_c, dim=0)).mean().item()
        cqm_clc_vals.append(cos_clc)
        # Grid
        z_grid = grid_canonicalize_batch(z0, T_net, v_c)
        cos_grid = (F.normalize(z_grid, dim=1) @ F.normalize(v_c, dim=0)).mean().item()
        cqm_grid_vals.append(cos_grid)
    return float(np.mean(cqm_clc_vals)), float(np.mean(cqm_grid_vals)), cqm_clc_vals, cqm_grid_vals

In [9]:
@torch.no_grad()
def compute_energy_landscape(
    z0:    torch.Tensor,
    T_net: HybridLieTransform,
    v_c:   torch.Tensor,
    t_range: tuple = (-np.pi, np.pi),
    n_points: int = 100
) -> tuple:
    """Sweep t continuously and record E(t) = 1 - cos(T(z0,t), v_c)."""
    ts = np.linspace(t_range[0], t_range[1], n_points)
    Es = []
    for t in ts:
        z_t = T_net(z0, float(t))
        E   = (1.0 - (F.normalize(z_t, dim=1) * F.normalize(v_c, dim=0)).sum(dim=1)).mean().item()
        Es.append(E)
    return ts, np.array(Es)

In [10]:
DARK, PANEL = "#0d0d0d", "#1a1a2e"
CMAP = ["#e94560", "#0f9b8e", "#f5a623", "#9b59b6", "#2ecc71"]
 
def _ax(ax, title="", xl="", yl=""):
    ax.set_facecolor(PANEL); ax.tick_params(colors="white")
    for s in ax.spines.values(): s.set_edgecolor("#444")
    if title: ax.set_title(title, color="white", fontsize=9)
    if xl:    ax.set_xlabel(xl, color="white", fontsize=8)
    if yl:    ax.set_ylabel(yl, color="white", fontsize=8)
 
def plot_canonicalization(
    angles, acc_baseline, acc_grid, acc_clc,
    cqm_clc, cqm_grid, cqm_clc_vals, cqm_grid_vals,
    test_angles, out_path
):
    fig = plt.figure(figsize=(20, 12), facecolor=DARK)
    fig.suptitle("Exp 3 — Lie-Guided Continuous Canonicalization (LGCC)\n"
                 "Novel: continuous angle optimization via Lie algebra energy minimization",
                 color="white", fontsize=12, fontweight="bold")
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)
    x  = np.array(angles)
 
    # Panel 1: Per-angle accuracy
    ax1 = fig.add_subplot(gs[0, :2])
    _ax(ax1, "Per-Angle Accuracy: Baseline vs Grid-Canon vs CLC-Canon",
        "Rotation Angle θ (°)", "Accuracy")
    for key, acc, col, ls in [
        ("MLP-NoAug",   acc_baseline, "#555",   "--"),
        ("Grid Canon",  acc_grid,     CMAP[1],  "-"),
        ("CLC (ours)",  acc_clc,      CMAP[0],  "-"),
    ]:
        y = np.array([acc[a] for a in angles])
        ax1.plot(x, y, ls, color=col, lw=2.2, label=f"{key}  [mean={y.mean():.4f}]")
    ax1.set_xticks(x); ax1.set_xticklabels([f"{a}°" for a in angles], color="white", fontsize=8)
    ax1.legend(facecolor=PANEL, labelcolor="white", fontsize=9)
    ax1.set_ylim(0, 1.05)
 
    # Panel 2: CQM per angle
    ax2 = fig.add_subplot(gs[0, 2])
    _ax(ax2, "Canonicalization Quality Metric (CQM)\ncos(z_canon, v_c)  — higher is better",
        "Input Rotation θ (°)", "CQM = cos alignment")
    ax2.plot(test_angles, cqm_clc_vals,  "o-", color=CMAP[0], lw=2, label=f"CLC  [μ={cqm_clc:.3f}]")
    ax2.plot(test_angles, cqm_grid_vals, "s--", color=CMAP[1], lw=2, label=f"Grid [μ={cqm_grid:.3f}]")
    ax2.axhline(1.0, color="white", ls=":", lw=1, alpha=0.4, label="perfect")
    ax2.set_xticks(test_angles); ax2.set_xticklabels([f"{a}°" for a in test_angles],
                                                      color="white", fontsize=7, rotation=45)
    ax2.legend(facecolor=PANEL, labelcolor="white", fontsize=8)
    ax2.set_ylim(0, 1.05)
 
    # Panel 3: summary bar
    ax3 = fig.add_subplot(gs[1, 0])
    _ax(ax3, "Mean Accuracy Summary", "", "Mean Acc (all θ)")
    methods = ["NoAug", "Grid-Canon", "CLC (ours)"]
    accs_m  = [np.mean(list(acc_baseline.values())),
               np.mean(list(acc_grid.values())),
               np.mean(list(acc_clc.values()))]
    cols = ["#555", CMAP[1], CMAP[0]]
    bars = ax3.bar(methods, accs_m, color=cols, alpha=0.85, edgecolor=DARK)
    for bar, v in zip(bars, accs_m):
        ax3.text(bar.get_x() + bar.get_width()/2, v + 0.005,
                 f"{v:.4f}", ha="center", color="white", fontsize=10, fontweight="bold")
    ax3.set_ylim(0, 1.1)
    ax3.tick_params(colors="white")
 
    # Panel 4: CLC vs Grid improvement per angle
    ax4 = fig.add_subplot(gs[1, 1:])
    _ax(ax4, "Per-Angle Accuracy Gain: CLC over Grid-Canon\n(Δ > 0 = continuous Lie beats discrete grid)",
        "Rotation Angle θ (°)", "Δ Accuracy (CLC − Grid)")
    delta = np.array([acc_clc[a] - acc_grid[a] for a in angles])
    bars2 = ax4.bar(x, delta, width=22,
                    color=[CMAP[0] if d >= 0 else "#e94560" for d in delta],
                    alpha=0.85, edgecolor=DARK)
    ax4.axhline(0, color="white", ls="--", lw=1)
    ax4.set_xticks(x); ax4.set_xticklabels([f"{a}°" for a in angles], color="white", fontsize=8)
    ax4.axhline(delta.mean(), color=CMAP[2], ls=":", lw=1.5,
                label=f"mean Δ = {delta.mean():+.4f}")
    ax4.legend(facecolor=PANEL, labelcolor="white", fontsize=8)
 
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    plt.close()
    print(f"  saved → {out_path}")
 
 
def plot_energy_landscape(
    vae, T_net, v_c, imgs_0, n_samples=4, out_path=""
):
    """Plot E(t) landscape for several test samples, showing energy wells."""
    rng = np.random.default_rng(7)
    idx = rng.choice(len(imgs_0), n_samples, replace=False)
    fig, axes = plt.subplots(2, n_samples, figsize=(n_samples * 4, 8), facecolor=DARK)
    fig.suptitle("Lie Canonicalization — Energy Landscape E(t) = 1 − cos(T(z,t), v_c)\n"
                 "Top: energy curve; Bottom: decoded image at minimum energy (canonical pose)",
                 color="white", fontsize=10, fontweight="bold")
 
    test_angles = [0, 90, 180, 270]
    angle_colors = [CMAP[i] for i in range(4)]
 
    for col, s_idx in enumerate(idx):
        ax_top = axes[0, col]
        ax_bot = axes[1, col]
        ax_top.set_facecolor(PANEL); ax_top.tick_params(colors="white")
        for s in ax_top.spines.values(): s.set_edgecolor("#444")
 
        # Plot E(t) for different input rotations
        for rot_deg, a_col in zip(test_angles, angle_colors):
            base = imgs_0[s_idx:s_idx+1]
            if rot_deg != 0:
                base = tvrotate(base, float(rot_deg),
                                interpolation=transforms.InterpolationMode.BILINEAR, fill=[0.])
            z0 = vae.encode(base.to(device))
            ts, Es = compute_energy_landscape(z0, T_net, v_c)
            ax_top.plot(np.degrees(ts), Es, color=a_col, lw=1.8,
                        label=f"input {rot_deg}°")
            # Mark minimum
            t_min = ts[np.argmin(Es)]
            ax_top.axvline(np.degrees(t_min), color=a_col, ls=":", lw=1, alpha=0.6)
 
        ax_top.set_xlabel("t (°)", color="white", fontsize=8)
        ax_top.set_ylabel("E(t)", color="white", fontsize=8)
        ax_top.set_title(f"Sample {col+1}", color="white", fontsize=9)
        if col == 0:
            ax_top.legend(facecolor=PANEL, labelcolor="white", fontsize=7)
 
        # Decoded canonical image (start from 0° and apply t*)
        z0 = vae.encode(imgs_0[s_idx:s_idx+1].to(device))
        z_canon, t_star, _ = lie_canonicalize_batch(z0, T_net, v_c, n_steps=50, lr=0.05)
        img_np = vae.decode(z_canon).clamp(0., 1.).cpu().squeeze().numpy()
        ax_bot.imshow(img_np, cmap="plasma", vmin=0, vmax=1)
        ax_bot.axis("off")
        ax_bot.set_title(f"Canonical  t*={np.degrees(t_star[0].item()):.1f}°",
                         color="white", fontsize=8)
 
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=DARK)
    plt.close()
    print(f"  saved → {out_path}")

In [11]:
def main():
    args = get_args()
    os.makedirs(args.out_dir, exist_ok=True)
    print(f"\n{'='*60}")
    print(f"  Bonus Task · Exp 3: Lie-Guided Continuous Canonicalization")
    print(f"  device = {device}")
    print(f"  output → {args.out_dir}")
    print(f"{'='*60}\n")
 
    # ── Load models ───────────────────────────────────────────────────────────
    vae    = load_vae(args.task1_ckpt, args.latent_dim)
    T_net  = load_hybrid_transform(args.task2_ckpt, args.latent_dim)
    clf    = load_clf(args.clf_ckpt, args.latent_dim, args.n_classes, key="noaug")
 
    # ── Load data ─────────────────────────────────────────────────────────────
    test_imgs, test_lbls, test_angs = load_h5(
        os.path.join(args.data_dir, "rotated_mnist_test.h5"))
    ANGLES  = sorted(test_angs.unique().tolist())
    DIGITS  = sorted(test_lbls.unique().tolist())
    dig2cls = {d: i for i, d in enumerate(DIGITS)}
 
    mask_0  = test_angs == 0
    imgs_0  = test_imgs[mask_0]
    lbls_0  = test_lbls[mask_0]
    print(f"  Test (0° only): {len(imgs_0)} samples  |  angles: {ANGLES}")
 
    # ── Canonical direction v_c ───────────────────────────────────────────────
    print(f"\n[1] Computing canonical direction v_c from {args.n_prototype_samples} samples …")
    v_c = compute_canonical_direction(vae, imgs_0, args.n_prototype_samples)
    print(f"  v_c computed  (norm={v_c.norm().item():.4f})")
 
    # ── Baseline (no canonicalization) ────────────────────────────────────────
    print("\n[2] Evaluating baseline (no canonicalization) …")
    def baseline_fn(imgs_chunk):
        return vae.encode(imgs_chunk)
    acc_baseline = evaluate_canon_strategy(baseline_fn, clf, vae, imgs_0, lbls_0,
                                           ANGLES, dig2cls, desc="baseline")
 
    # ── Grid canonicalization ─────────────────────────────────────────────────
    print("\n[3] Evaluating grid canonicalization (discrete 30° steps) …")
    def grid_fn(imgs_chunk):
        z0 = vae.encode(imgs_chunk)
        return grid_canonicalize_batch(z0, T_net, v_c, [i*30 for i in range(12)])
    acc_grid = evaluate_canon_strategy(grid_fn, clf, vae, imgs_0, lbls_0,
                                       ANGLES, dig2cls, desc="grid-canon")
 
    # ── CLC: Continuous Lie Canonicalization ──────────────────────────────────
    print(f"\n[4] Evaluating CLC (continuous, {args.n_canon_steps} Adam steps per sample) …")
    def clc_fn(imgs_chunk):
        z0 = vae.encode(imgs_chunk)
        z_canon, _, _ = lie_canonicalize_batch(
            z0, T_net, v_c, args.n_canon_steps, args.canon_lr)
        return z_canon
    acc_clc = evaluate_canon_strategy(clc_fn, clf, vae, imgs_0, lbls_0,
                                      ANGLES, dig2cls, desc="CLC")
 
    # ── Canonicalization Quality Metric ──────────────────────────────────────
    print("\n[5] Computing Canonicalization Quality Metric (CQM) …")
    test_angles_cqm = [0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330]
    cqm_clc, cqm_grid, cqm_clc_vals, cqm_grid_vals = canonicalization_quality_metric(
        T_net, vae, imgs_0, v_c, args.n_canon_steps, args.canon_lr)
 
    # ── Summary ───────────────────────────────────────────────────────────────
    mean_base = np.mean(list(acc_baseline.values()))
    mean_grid = np.mean(list(acc_grid.values()))
    mean_clc  = np.mean(list(acc_clc.values()))
 
    print(f"\n{'='*60}")
    print(f"  RESULTS SUMMARY — Exp 3")
    print(f"{'='*60}")
    print(f"  MLP-NoAug (no canon)     mean acc = {mean_base:.4f}")
    print(f"  Grid canonicalization    mean acc = {mean_grid:.4f}  Δ = {mean_grid-mean_base:+.4f}")
    print(f"  CLC (ours, continuous)   mean acc = {mean_clc:.4f}  Δ = {mean_clc-mean_base:+.4f}")
    print(f"\n  CQM (alignment with v_c):")
    print(f"    CLC  = {cqm_clc:.4f}")
    print(f"    Grid = {cqm_grid:.4f}")
    print(f"    CLC advantage over Grid = {cqm_clc - cqm_grid:+.4f}")
 
    # ── Save ──────────────────────────────────────────────────────────────────
    print("\n[6] Saving results and figures …")
    np.savez(os.path.join(args.out_dir, "canon_results.npz"),
             angles        = ANGLES,
             acc_baseline  = [acc_baseline[a] for a in ANGLES],
             acc_grid      = [acc_grid[a]      for a in ANGLES],
             acc_clc       = [acc_clc[a]       for a in ANGLES],
             cqm_clc       = cqm_clc,
             cqm_grid      = cqm_grid,
             cqm_clc_vals  = cqm_clc_vals,
             cqm_grid_vals = cqm_grid_vals,
             test_angles_cqm = test_angles_cqm)
 
    plot_canonicalization(
        ANGLES, acc_baseline, acc_grid, acc_clc,
        cqm_clc, cqm_grid, cqm_clc_vals, cqm_grid_vals, test_angles_cqm,
        os.path.join(args.out_dir, "fig_canonicalization.png"))
 
    plot_energy_landscape(
        vae, T_net, v_c, imgs_0, n_samples=4,
        out_path=os.path.join(args.out_dir, "fig_energy_landscape.png"))
 
    print(f"\n✅  Exp 3 complete.  Outputs → {args.out_dir}/")
    print(f"    CLC mean accuracy = {mean_clc:.4f}  "
          f"(+{mean_clc - mean_base:.4f} over no-invariance baseline)")

In [12]:
if __name__ == "__main__":
    main()


  Bonus Task · Exp 3: Lie-Guided Continuous Canonicalization
  device = cuda
  output → ./bonus_outputs/exp3

  ✓ VAE loaded (epoch 47)
✓ HybridLieTransform loaded correctly
  ✓ Classifier ('noaug') loaded
  loading rotated_mnist_test.h5 ... done (0.7s)
  Test (0° only): 2167 samples  |  angles: [0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330]

[1] Computing canonical direction v_c from 200 samples …
  v_c computed  (norm=1.0000)

[2] Evaluating baseline (no canonicalization) …


  eval baseline:   0%|          | 0/12 [00:00<?, ?it/s]


[3] Evaluating grid canonicalization (discrete 30° steps) …


  eval grid-canon:   0%|          | 0/12 [00:00<?, ?it/s]


[4] Evaluating CLC (continuous, 5 Adam steps per sample) …


  eval CLC:   0%|          | 0/12 [00:00<?, ?it/s]


[5] Computing Canonicalization Quality Metric (CQM) …

  RESULTS SUMMARY — Exp 3
  MLP-NoAug (no canon)     mean acc = 0.8496
  Grid canonicalization    mean acc = 0.5477  Δ = -0.3019
  CLC (ours, continuous)   mean acc = 0.9812  Δ = +0.1316

  CQM (alignment with v_c):
    CLC  = -0.4094
    Grid = -0.0442
    CLC advantage over Grid = -0.3653

[6] Saving results and figures …
  saved → ./bonus_outputs/exp3/fig_canonicalization.png
  saved → ./bonus_outputs/exp3/fig_energy_landscape.png

✅  Exp 3 complete.  Outputs → ./bonus_outputs/exp3/
    CLC mean accuracy = 0.9812  (+0.1316 over no-invariance baseline)
